# 飞书 (Feishu/Lark) API 全流程验证

> **目标**：验证飞书 Open API 的完整能力链路
> **覆盖**：认证 → 创建 → 追加(文本/标题/列表/代码/公式/表格/图片) → 读取 → 块操作 → 搜索 → 清理
> **认证**：App ID + App Secret → tenant_access_token

In [ ]:
from feishu_client import FeishuClient, md_to_blocks, make_text_block, make_heading_block, make_code_block, extract_text_from_block, block_type_name
from pathlib import Path
from datetime import datetime
import json

# 初始化客户端（自动从 .env 读取 FEISHU_APP_ID / FEISHU_APP_SECRET）
client = FeishuClient()

# 健康检查
health = client.health_check()
print(f"[{'OK' if health['ok'] else 'FAIL'}] FeishuClient initialized")
print(f"       Token valid: {health['token_valid']}")
print(f"       Expire in: {health['expire_in']}s")

## 1. 创建文档

验证 `POST /docx/v1/documents` 创建 docx 格式文档。

In [ ]:
# 创建测试文档
title = f"API验证 - {datetime.now().strftime('%H:%M:%S')}"
result = client.api("POST", "/docx/v1/documents", json_data={"title": title})
document_id = result["document"]["document_id"]

print(f"[OK] Created doc: {title}")
print(f"     document_id: {document_id}")
print(f"     URL: https://open.feishu.cn/docx/{document_id}")

## 2. 追加文本内容（md_to_blocks 方式）

使用 `md_to_blocks()` 将 Markdown 文本转换为飞书 Block 格式，然后追加到文档末尾。

In [ ]:
content = """## 文本追加测试

这是一段普通文本，用于验证 feishu_doc_append 的 content 模式。

- 支持无序列表
- 自动转换为飞书 bullet block

1. 有序列表项1
2. 有序列表项2

> 引用块测试

段落之间需要空行分隔。"""

blocks = md_to_blocks(content)
print(f"[INFO] Converted to {len(blocks)} blocks")
for b in blocks:
    print(f"  {block_type_name(b['block_type'])}: {extract_text_from_block(b)[:30]}...")

result = client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": blocks}
)
print(f"[OK] Appended {len(blocks)} blocks")

## 3. 追加代码块

验证 `block_type: 14` code block 的创建。

In [ ]:
code_blocks = [
    make_heading_block("代码块测试", level=2),
    make_text_block("Python 示例代码:"),
    make_code_block("def hello():\n    print('Hello, Feishu!')\n\nhello()"),
]
result = client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": code_blocks}
)
print("[OK] Appended code block")

## 4. 追加数学公式

飞书原生支持公式，通过 `text_element_style.formula` 或 `inline_formula` 设置。

**注意**：公式内容必须是 KaTeX 语法，`\frac` 等命令需要双反斜杠 `\\frac`。

In [ ]:
def make_formula_block(latex, inline=False):
    return {
        "block_type": 2,
        "text": {
            "elements": [{
                "text_run": {
                    "content": latex,
                    "text_element_style": {
                        "inline_formula" if inline else "formula": True
                    }
                }
            }]
        }
    }

formula_blocks = [
    make_heading_block("数学公式测试", level=2),
    make_text_block("一元二次方程求根公式:"),
    make_formula_block("x = \\frac{-b \\pm \\sqrt{b^2-4ac}}{2a}"),
]
result = client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": formula_blocks}
)
print("[OK] Appended formula blocks")

## 5. 追加表格

直接构造 `block_type: 31` table + `block_type: 32` table_cell 块。

In [ ]:
table_block = {
    "block_type": 31,
    "table": {
        "property": {
            "row_size": 3,
            "column_size": 3,
            "merge_type": 0,
            "header_row": True,
            "header_column": False
        },
        "cells": [
            {"block_type": 32, "table_cell": {"children": [make_text_block("姓名")]}},
            {"block_type": 32, "table_cell": {"children": [make_text_block("年龄")]}},
            {"block_type": 32, "table_cell": {"children": [make_text_block("城市")]}},
            {"block_type": 32, "table_cell": {"children": [make_text_block("Alice")]}},
            {"block_type": 32, "table_cell": {"children": [make_text_block("25")]}},
            {"block_type": 32, "table_cell": {"children": [make_text_block("北京")]}},
            {"block_type": 32, "table_cell": {"children": [make_text_block("Bob")]}},
            {"block_type": 32, "table_cell": {"children": [make_text_block("30")]}},
            {"block_type": 32, "table_cell": {"children": [make_text_block("上海")]}},
        ]
    }
}
result = client.api(
    "POST",
    f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
    json_data={"children": [make_heading_block("表格测试", level=2), table_block]}
)
print("[OK] Appended table block")

## 6. 上传图片

验证 `POST /drive/v1/medias/upload_all` 上传图片到文档素材库，返回 `file_token`。

In [ ]:
from PIL import Image
import io
import base64

# 创建测试图片
img = Image.new('RGB', (400, 200), color=(73, 109, 137))
from PIL import ImageDraw, ImageFont
draw = ImageDraw.Draw(img)
try:
    font = ImageFont.truetype("arial.ttf", 24)
except:
    font = ImageFont.load_default()
draw.text((20, 80), "Test Image for Feishu", fill=(255, 255, 255), font=font)

buf = io.BytesIO()
img.save(buf, format='PNG')
img_bytes = buf.getvalue()

print(f"[INFO] Image size: {len(img_bytes)} bytes")

upload_result = client.request(
    "POST",
    "/drive/v1/medias/upload_all",
    files={"file": ("test.png", img_bytes, "image/png")},
    data={
        "file_name": "test.png",
        "parent_type": "doc_image",
        "parent_node": document_id,
        "size": str(len(img_bytes)),
    }
)

if upload_result.get("code", -1) == 0:
    file_token = upload_result["data"]["file_token"]
    print(f"[OK] Image uploaded, file_token: {file_token}")
else:
    print(f"[WARN] Upload failed: {upload_result.get('msg')}")
    file_token = None

## 7. 插入图片块

使用 `block_type: 27` image + `token: file_token` 插入图片到文档。

In [ ]:
if 'file_token' in globals() and file_token:
    image_block = {
        "block_type": 27,
        "image": {"token": file_token}
    }
    result = client.api(
        "POST",
        f"/docx/v1/documents/{document_id}/blocks/{document_id}/children",
        json_data={"children": [make_heading_block("图片测试", level=2), image_block]}
    )
    print("[OK] Inserted image block")
else:
    print("[SKIP] No file_token, skip image block")

## 8. 读取文档

验证 `GET /docx/v1/documents/{id}/content` 读取纯文本。

In [ ]:
read_result = client.api("GET", f"/docx/v1/documents/{document_id}/content")
texts = []
for item in read_result.get("content", []):
    t = item.get("text", "")
    if t:
        texts.append(t)
print(f"[OK] Read doc, extracted {len(texts)} text segments")
print("--- Content preview ---")
for t in texts[:10]:
    print(t[:80])

## 9. 获取块结构

验证 `GET /docx/v1/documents/{id}/blocks` 获取文档块列表，用于后续更新/删除。

In [ ]:
blocks_result = client.api("GET", f"/docx/v1/documents/{document_id}/blocks", params={"page_size": 500})
block_items = blocks_result.get("items", [])

print(f"[OK] Got {len(block_items)} blocks")
print(f"{'Block ID':<30} {'Type':<15} {'Preview'}")
print("-" * 70)
for b in block_items[:15]:
    bt = b.get("block_type", 0)
    bid = b.get("block_id", "")[:28]
    preview = extract_text_from_block(b)[:30]
    print(f"{bid:<30} {block_type_name(bt):<15} {preview}")

## 10. 更新指定块

验证 `PUT /docx/v1/documents/{id}/blocks/{block_id}` 更新块内容。

In [ ]:
text_blocks = [b for b in block_items if b.get("block_type") == 2]
if text_blocks:
    target = text_blocks[0]
    target_id = target["block_id"]
    update_result = client.api(
        "PUT",
        f"/docx/v1/documents/{document_id}/blocks/{target_id}",
        json_data={
            "replace_block": {
                "block_type": 2,
                "text": {
                    "elements": [{
                        "text_run": {
                            "content": "[已更新] 这段文本已被 feishu_doc_update_block 修改",
                            "text_element_style": {}
                        }
                    }]
                }
            }
        }
    )
    print(f"[OK] Updated block {target_id}")
else:
    print("[SKIP] No text block found")

## 11. 搜索文档

验证 `POST /suite/docs-api/search/object` 搜索云空间文档。

In [ ]:
try:
    search_result = client.api(
        "POST",
        "/suite/docs-api/search/object",
        json_data={"search_key": "API", "count": 5}
    )
    docs = search_result.get("docs_entities", [])
    print(f"[OK] Found {len(docs)} docs")
    for d in docs[:3]:
        print(f"  - {d.get('title', 'N/A')} ({d.get('type', 'N/A')})")
except Exception as e:
    print(f"[WARN] Search failed: {e}")

## 12. 搜索用户

验证 `POST /contact/v3/users/batch_get_id` 用户查找。

**注意**：需要应用有通讯录权限。

In [ ]:
try:
    user_result = client.api(
        "POST",
        "/contact/v3/users/batch_get_id",
        json_data={"emails": ["test@example.com"]},
        params={"user_id_type": "open_id"}
    )
    users = user_result.get("user_list", [])
    print(f"[OK] Found {len(users)} users")
    for u in users:
        print(f"  - {u.get('user_id', 'N/A')}")
except Exception as e:
    print(f"[WARN] User search failed: {e}")

## 13. 删除测试块

验证 `DELETE /docx/v1/documents/{id}/blocks/{block_id}` 删除指定块。

In [ ]:
if len(block_items) > 1:
    last_block = block_items[-1]
    last_id = last_block["block_id"]
    del_result = client.api(
        "DELETE",
        f"/docx/v1/documents/{document_id}/blocks/{last_id}"
    )
    print(f"[OK] Deleted last block {last_id}")
else:
    print("[SKIP] Too few blocks to delete")

## 14. 验证总结

In [ ]:
print("=" * 60)
print("飞书 API 全流程验证完成")
print("=" * 60)
print(f"测试文档 ID: {document_id}")
print(f"文档链接: https://open.feishu.cn/docx/{document_id}")
print()
print("已验证功能:")
print("  [OK] 认证 (tenant_access_token)")
print("  [OK] 创建文档")
print("  [OK] 追加文本/标题/列表/引用 (content 模式)")
print("  [OK] 追加代码块")
print("  [OK] 追加数学公式")
print("  [OK] 追加表格")
print("  [OK] 上传图片")
print("  [OK] 插入图片块")
print("  [OK] 读取文档")
print("  [OK] 获取块结构")
print("  [OK] 更新块")
print("  [OK] 搜索文档")
print("  [OK] 搜索用户")
print("  [OK] 删除块")
print()
print("注意: 测试文档未自动删除，请手动清理")